In [2]:
%load_ext autoreload
%autoreload 2

import os
import numpy as np
import tensorflow as tf
import csv
import pandas as pd
import model as md
import utils as ut

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [3]:
# configurations
# inFilePath = "/asic/projects/C/CMS_PIX_28/pixelAV_datasets/unshuffled_DO_NOT_DELETE/initial_studies/dataset14/unflipped"
# inFilePath = "/asic/projects/C/CMS_PIX_28/pixelAV_datasets/unshuffled_DO_NOT_DELETE/dataset_2sNoise/dataset_2sNoise_50x12P5_16x16_100e-sigma_parquets/unflipped"
inFilePath = "/asic/projects/C/CMS_PIX_28/pixelAV_datasets/unshuffled_DO_NOT_DELETE/dataset_2s16x16_centeredIncidence/unflipped"
outDir="./tmp_16x16_hlevOptimized"
confs = [
    # {"qm_charge_levels" : [200, 1600, 2400], "qm_quant_values" : [0, 1, 2, 3], "zero_pad": False},
    # {"qm_charge_levels" : [300, 1600, 2400], "qm_quant_values" : [0, 1, 2, 3], "zero_pad": False},
    # {"qm_charge_levels" : [350, 1600, 2400], "qm_quant_values" : [0, 1, 2, 3], "zero_pad": False},
    # {"qm_charge_levels" : [400, 800, 1200], "qm_quant_values" : [0, 1, 2, 3], "zero_pad": False},
    # {"qm_charge_levels" : [500, 1600, 2400], "qm_quant_values" : [0, 1, 2, 3], "zero_pad": False},
    # {"qm_charge_levels" : [600, 1600, 2400], "qm_quant_values" : [0, 1, 2, 3], "zero_pad": False},
    # {"qm_charge_levels" : [700, 1400, 2100], "qm_quant_values" : [0, 1, 2, 3], "zero_pad": False},
    # {"qm_charge_levels" : [800, 1600, 2400], "qm_quant_values" : [0, 1, 2, 3], "zero_pad": False},
    # {"qm_charge_levels" : [900, 1600, 2400], "qm_quant_values" : [0, 1, 2, 3], "zero_pad": False},
    {"qm_charge_levels" : [1000, 2000, 3000], "qm_quant_values" : [0, 1, 2, 3], "zero_pad": False},
#     {"qm_charge_levels" : [1100, 1600, 2400], "qm_quant_values" : [0, 1, 2, 3], "zero_pad": False},
#     {"qm_charge_levels" : [1200, 1600, 2400], "qm_quant_values" : [0, 1, 2, 3], "zero_pad": False},
#     {"qm_charge_levels" : [1300, 1600, 2400], "qm_quant_values" : [0, 1, 2, 3], "zero_pad": False},
#     {"qm_charge_levels" : [1600, 2000, 2400], "qm_quant_values" : [0, 1, 2, 3], "zero_pad": False},
]


# create model
shape = 16 # y-profile ... why is this 16 and not 8?
nb_classes = 3 # positive low pt, negative low pt, high pt
first_dense = 58 # shape of first dense layer

# keras model
model_file = "/asic/projects/C/CMS_PIX_28/dshekar/filter/model_pipeline/tmp_16x16_hlevOptimized/trainedModel_1000_2000_3000/model0/models/tmp_padded_noscaling_keras_d58model_0.h5"
# model_file = "/fasic_home/gdg/research/projects/CMS_PIX_28/directional-pixel-detectors/multiclassifier/for_testing/models/ds8l6_padded_noscaling_keras_d58model_9.h5"
# model_file = "/fasic_home/gdg/research/projects/CMS_PIX_28/directional-pixel-detectors/multiclassifier/models/ds8l6_padded_noscaling_keras_d58model.h5"
model = md.CreateModel(shape, nb_classes, first_dense, model_file = model_file)

# qkeras model
qmodel_file =  "/asic/projects/C/CMS_PIX_28/dshekar/filter/model_pipeline/tmp_16x16_hlevOptimized/trainedModel_1000_2000_3000/model0/models/tmp_padded_noscaling_qkeras_foldbatchnorm_d58w4a8model_0.h5"
# qmodel_file =  "/fasic_home/gdg/research/projects/CMS_PIX_28/directional-pixel-detectors/multiclassifier/for_testing/models/ds8l6_padded_noscaling_qkeras_foldbatchnorm_d58w4a8model_9.h5"
# qmodel_file = "/fasic_home/gdg/research/projects/CMS_PIX_28/directional-pixel-detectors/multiclassifier/models/ds8l6_padded_noscaling_qkeras_foldbatchnorm_d58w4a8model.h5"

Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input (InputLayer)          [(None, 16)]              0         
                                                                 
 dense1 (Dense)              (None, 58)                986       
                                                                 
 batch_normalization (Batch  (None, 58)                232       
 Normalization)                                                  
                                                                 
 relu1 (Activation)          (None, 58)                0         
                                                                 
 dense2 (Dense)              (None, 3)                 177       
                                                                 
 linear (Activation)         (None, 3)                 0         
                                                             

2026-05-18 18:17:38.020954: I tensorflow/compiler/xla/service/service.cc:168] XLA service 0x55d1a53e9c00 initialized for platform Host (this does not guarantee that XLA will be used). Devices:
2026-05-18 18:17:38.021003: I tensorflow/compiler/xla/service/service.cc:176]   StreamExecutor device (0): Host, Default Version


In [4]:
# load the parquet data, filter on y-profile bin, and quantize (all done in loadParquetData). Previously parqueted data was loaded and filtered on y-local in the next code-block. However this was moved to loadParquetData to avoid loading the entire dataset into memory and quantizing.
for conf in confs:
    if not os.path.exists(outDir):
        os.makedirs(outDir)
    print(f"Running with configuration: {conf}")
    conf["qm"] = ut.loadParquetData(inFilePath=inFilePath, qm_charge_levels = conf["qm_charge_levels"], qm_quant_values = conf["qm_quant_values"], zero_pad = conf["zero_pad"],outDir=outDir,useYProfilePulsing=False) # Create pulsing pattern using Yprofile. If False, use the recon2D data directly, DS (20Oct25)

Running with configuration: {'qm_charge_levels': [1000, 2000, 3000], 'qm_quant_values': [0, 1, 2, 3], 'zero_pad': False}


  1%|          | 1/82 [00:00<00:11,  7.26it/s]

NOTE!!  Quantizing data with levels: [1000, 2000, 3000] and values: [0, 1, 2, 3]


  2%|▏         | 2/82 [00:00<00:10,  7.64it/s]

NOTE!!  Quantizing data with levels: [1000, 2000, 3000] and values: [0, 1, 2, 3]


...

NOTE!!  Quantizing data with levels: [1000, 2000, 3000] and values: [0, 1, 2, 3]
NOTE!!  Quantizing data with levels: [1000, 2000, 3000] and values: [0, 1, 2, 3]


 99%|█████████▉| 81/82 [00:10<00:00,  7.40it/s]

NOTE!!  Quantizing data with levels: [1000, 2000, 3000] and values: [0, 1, 2, 3]
NOTE!!  Quantizing data with levels: [1000, 2000, 3000] and values: [0, 1, 2, 3]


100%|██████████| 82/82 [00:10<00:00,  7.57it/s]


NOTE!!  Quantizing data with levels: [1000, 2000, 3000] and values: [0, 1, 2, 3]
(169607,) (169607,) (169607,)
169607
Creating yprofiles


In [5]:
y_local_bins = np.linspace(-8.1, 8.1, 13)
bin_number = 6
y_local_min, y_local_max = y_local_bins[bin_number], y_local_bins[bin_number + 1]
for conf in confs:
    # Save asic compout
    outDict = conf["qm"]
    # create compout of y-local subset
    if conf["qm"]["outDir"] is not None:
        compout_file_name = os.path.join(conf["qm"]["outDir"], f'compouts_ylocal_{y_local_min:.2f}_{y_local_max:.2f}.csv')
        if conf["qm"]["recon2D"] is not None:
            ut.recon2DToCompoutWrite(conf["qm"]["recon2D"], compout_file_name)
        else:
            # set flip to False if you want the compout to program the other side of asic pixel matrix (DS, Oct 19 2025)
            ut.yprofileToCompoutWrite(conf["qm"]["yprofiles"], compout_file_name, flip=True)
        outDict["compout_file_name"] = compout_file_name

for conf in confs:
    conf["filtered_qm"] = conf["qm"]

Making compout of y-local subset
   writing to file: ./tmp_16x16_hlevOptimized/1000_2000_3000/compouts_ylocal_0.00_1.35.csv
   done!


In [6]:
# keras model
model = md.CreateModel(shape, nb_classes, first_dense, model_file = model_file)

# qkeras model
qmodel = md.CreateQModel(shape, model_file=qmodel_file)
# generate hls model
ut.gen_hls_model(qmodel, output_dir=outDir)
# prepare weights for ASIC
ut.prepareWeights(os.path.join(outDir, "firmware/weights/"))

Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input (InputLayer)          [(None, 16)]              0         
                                                                 
 dense1 (Dense)              (None, 58)                986       
                                                                 
 batch_normalization (Batch  (None, 58)                232       
 Normalization)                                                  
                                                                 
 relu1 (Activation)          (None, 58)                0         
                                                                 
 dense2 (Dense)              (None, 3)                 177       
                                                                 
 linear (Activation)         (None, 3)                 0         
                                                             

/mnt/local/CMSPIX28/miniforge3/envs/p3-11-11-new/lib/python3.11/site-packages/keras/src/engine/training.py:3000: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


Done


'./tmp_16x16_hlevOptimized/firmware/weights/b5_w5_b2_w2_pixel_bin.csv'

In [7]:
# Compute loss and accuracy manually
def getLA(y, predictions, loss_fn, acc_metric=tf.keras.metrics.SparseCategoricalAccuracy()):
    loss = loss_fn(y, predictions).numpy()
    acc_metric.update_state(y, predictions)
    accuracy = acc_metric.result().numpy()
    return loss, accuracy

# evaluating
verbose = 1
batch_size = 2048

# loop over the confs
for conf in confs:
    
    # make predictions
    for m, name in zip([model, qmodel], ["keras", "qkeras"]):
        print(f"Evaluating {name} model...")
        conf[f"{name}_predictions"] = m.predict(conf["qm"]["yprofiles"], batch_size = batch_size, verbose=verbose)
        # predictions = np.argmax(predictions, axis=1)
        predFileName = os.path.join(conf["qm"]["outDir"], f"{name}_predictions.npy")
        np.save(predFileName, conf[f"{name}_predictions"])
        model_loss, model_acc = getLA(conf["qm"]["clslabels"], conf[f"{name}_predictions"], md.custom_loss_function)
        print(f"Finished evaluating {name} model with loss: {model_loss}, accuracy: {model_acc}, predictions saved to {predFileName}")
        print()


Evaluating keras model...


83/83 [==============================] - 0s 653us/step
Finished evaluating keras model with loss: 0.6236894130706787, accuracy: 0.7648210525512695, predictions saved to ./tmp_16x16_hlevOptimized/1000_2000_3000/keras_predictions.npy

Evaluating qkeras model...
83/83 [==============================] - 0s 1ms/step
Finished evaluating qkeras model with loss: 0.5020045638084412, accuracy: 0.7775121331214905, predictions saved to ./tmp_16x16_hlevOptimized/1000_2000_3000/qkeras_predictions.npy



In [8]:
test_dir = "/fasic_home/gdg/research/projects/CMS_PIX_28/directional-pixel-detectors/multiclassifier/data/ds8_only/dec6_ds8_quant/QuantizedInputTestSetLocal6.csv"
test_labels = "/fasic_home/gdg/research/projects/CMS_PIX_28/directional-pixel-detectors/multiclassifier/data/ds8_only/dec6_ds8_quant/TestSetLabelLocal6.csv"
data = pd.read_csv(test_dir)
data_padded = pd.concat([data, pd.DataFrame(0, index=data.index, columns=['13', '14', '15'])], axis=1)
labels = pd.read_csv(test_labels, header=None, skiprows=1)

# Convert to NumPy arrays
ds8_test = data_padded.to_numpy()
# d28_test_labels = labels.to_numpy()
d28_test_labels = labels[0].to_numpy()  # Assuming labels are in the first column
# Ensure labels are integers
d28_test_labels = d28_test_labels.astype(int)

# Get predictions
predictions = qmodel.predict(ds8_test, batch_size=batch_size, verbose=verbose)
# print(predictions)
predictions = np.argmax(predictions, axis=1)
# predictions_npy = predictions.to_numpy()

model_loss, model_acc = getLA(d28_test_labels, predictions, md.custom_loss_function)
print(f"Finished evaluating qmodel with loss: {model_loss}, accuracy: {model_acc}.")



# def getLA(y, predictions, loss_fn, acc_metric=tf.keras.metrics.SparseCategoricalAccuracy()):
#     loss = loss_fn(y, predictions).numpy()
#     acc_metric.update_state(y, predictions)
#     accuracy = acc_metric.result().numpy()
#     return loss, accuracy


7/7 [==============================] - 0s 1ms/step


ValueError: `labels.shape` must equal `logits.shape` except for the last dimension. Received: labels.shape=(13983,) and logits.shape=(1, 13983)